# Power traces acquisition of AES HW

In [ ]:
import sys
import os
import time
from pathlib import Path

## Project paths
This block robustly detects the project root (`DOJO_ROOT`) starting from either the script location (`__file__`) or the current working directory (for notebooks). From there it defines all relevant subdirectories (AES sources, SCA scripts, X-HEEP, traces, plots, cache), ensures output folders exist, and adds the local source paths to `sys.path` .

In [ ]:
def find_project_root(start: Path, markers=("fusesoc.conf", ".dojo_root")) -> Path:
    current = start
    while current != current.parent:
        if any((current / m).exists() for m in markers):
            return current
        current = current.parent
    raise RuntimeError(
        f"Could not find project root (looked for markers: {markers}). "
        "Please ensure you are inside the Side-Channel-Dojo repository."
    )

# Establish directories dynamically
try:
    SCRIPT_DIR = Path(__file__).resolve().parent
except NameError:
    SCRIPT_DIR = Path.cwd()

DOJO_ROOT = find_project_root(SCRIPT_DIR)

AES_PY_DIR    = DOJO_ROOT / "sw" / "ciphers" / "AES_python"
SCA_DIR       = DOJO_ROOT / "sw" / "sca_scripts"
UTILS_DIR     = DOJO_ROOT / "sw" / "sca_scripts" / "utils"
HW_DIR        = DOJO_ROOT / "hw"
TRACESET_DIR  = DOJO_ROOT / "sw" / "traceset" / "AES" / "hw"

# Ensure traceset directory exists
TRACESET_DIR.mkdir(parents=True, exist_ok=True)

# Make local modules importable
sys.path.insert(0, str(AES_PY_DIR))
sys.path.insert(0, str(SCA_DIR))
sys.path.insert(0, str(UTILS_DIR))

## Imports

In [ ]:
import numpy as np
from tqdm.auto import tqdm
import chipwhisperer as cw
from chipwhisperer.common.traces import Trace
from pico_api import PS5000aWrapper
from CW305_api import CW305Wrapper
from AES_golden import AES_golden_model

## Configuration

In [ ]:
# ---------------------------------------------------------------------------
# Configuration parameters
# ---------------------------------------------------------------------------
tested_sbox = "sbox_rijandael"
sbox_id = tested_sbox.replace("sbox_", "")

capture = True          # True to capture from PicoScope, False to load existing
save_traces = True
n_trc = 5000            # Number of traces

# Output control
traces_overlapped_plot  = True   # Plot overlapped power traces
save_plot               = False  # Save plots to disk


bitstream_path = HW_DIR / "fpga" / "bitstream" / "aes" / "aes_single_round" / f"cw305_top_{sbox_id}_lut.bit"
bitstream = str(bitstream_path)
verilog_defines_path = (
    HW_DIR
    / "crypto_asic"
    / "aes"
    / "fpga"
    / "cw305_aes_defines.v"
)
verilog_defines = str(verilog_defines_path)

project_file_path = TRACESET_DIR / f"CW305_hw_AES_{sbox_id}.cwp"
project_file = str(project_file_path)

def _yn(flag: bool) -> str:
    return "yes" if flag else "no"

print("\n================= CONFIGURATION =================")
print(f"DOJO_ROOT           : {DOJO_ROOT}")
print()
print("Target")
print(f"  Tested S-box      : {tested_sbox}")
print(f"  Action            : {'Capture' if capture else 'Load dataset'}")
print()
print("Paths")
print(f"  Bitstream         : {bitstream}")
print(f"  CW project (.cwp) : {project_file}")
print(f"  Traceset dir      : {TRACESET_DIR}")
print("=================================================\n")

## Board preparation

In [ ]:
def prepare_board():
    """
    Prepare the CW305 board for the attack:
      1. Initialize the PicoScope.
      2. Initialize CW305 with the given bitstream and Verilog defines.
      3. Configure PLL and clock settings.

    Returns:
        (ps, cw305) handles for scope and board.
    """
    try:
        # 1) Initialize PicoScope
        ps = PS5000aWrapper()
        ps.get_unitInfo()
        ps.scope_setup(obs_time=3.225E-6, nSamples=1260)
        num_samples = ps.nSamples
        
        # 2) Initialize CW305 with required parameters
        cw305 = cw.target(
            None,
            cw.targets.CW305,
            bsfile=bitstream,
            force=True,
            slurp=True,
            defines_files=[verilog_defines],
        )

        cw305.vccint_set(1.0)
        cw305.pll.pll_enable_set(True)        # enable PLL chip
        cw305.pll.pll_outenable_set(False, 0) # disable PLL 0
        cw305.pll.pll_outenable_set(True, 1)  # enable PLL 1
        cw305.pll.pll_outenable_set(False, 2) # disable PLL 2
        cw305.pll.pll_outfreq_set(10e6, 1)    # PLL1 frequency set to 10 MHz

        # Disable USB clock (optional, reduces noise in the power traces)
        cw305.clkusbautooff = True
        # Idle time between captures
        cw305.clksleeptime = 1

        # 3) Set the clock source via FPGA register
        cw305.fpga_write(cw305.REG_CLKSETTINGS, data=bytearray([0x01]))


        return ps, cw305

    except ModuleNotFoundError as e:
        print(e)

## Online phase

In [ ]:
# ---------------------------------------------------------------------------
# Execution Flow
# ---------------------------------------------------------------------------

if not capture:
    try:
        print(f"[OFFLINE] Loading ChipWhisperer project from {project_file}...")
        project = cw.open_project(project_file)
        traces = np.array([wave for wave in project.waves])
        plaintexts = np.array([pt for pt in project.textins])
        ciphertexts = np.array([ct for ct in project.textouts])
        keys = np.array([k for k in project.keys])
        print(f"Loaded {traces.shape[0]} traces with {traces.shape[1]} samples each.")
        project.close()
    except Exception as e:
        print(f"Error loading project: {e}")

else:
    print("[ONLINE] Preparing capture board...")
    try:
        ps, cw305 = prepare_board()
        print("\n[ONLINE] Board initialized successfully.")
        print("[ONLINE] Picoscope settings:")
        print(ps.get_scopeSettings())
        print(f"[ONLINE] Sampling Interval : {ps.get_samplingInterval()} s\n")
    except Exception as e:
        print(f"[ERROR] Failed to prepare board (PicoScope + CW305): {e}")
        raise

    # Key, plaintext generation and golden model setup
    ktp = cw.ktp.Basic()
    key, plaintext = ktp.next()
    cipher = AES_golden_model()

    formatted_key = ''.join(format(el, '02x') for el in key)
    print(f"[ONLINE] Fixed key used: {[hex(subkey) for subkey in key]}")

    cw305.set_key(key)

    # Dummy capture inline (AC coupling bug mitigation)
    cw305.fpga_write(cw305.REG_CRYPT_TEXTIN, plaintext[::-1])
    ps.runBlock()
    time.sleep(0.05)
    cw305.usb_trigger_toggle()
    ps.waitReady()

    try:
        TRACESET_DIR.mkdir(parents=True, exist_ok=True)
        if save_traces:
            print("[ONLINE] Trace storage enabled. Creating CW project...")
            project_dir = os.path.dirname(project_file)
            os.makedirs(project_dir, exist_ok=True)
            project = cw.create_project(project_file, overwrite=True)
        else:
            print("[ONLINE] Trace storage disabled (save_traces = False).")
            project = None

        print("[ONLINE] Starting trace capture...")
        for i in tqdm(range(n_trc), desc="Capturing traces"):
            
            # Write plaintext to target (endianness reversed for HW AES IP)
            cw305.fpga_write(cw305.REG_CRYPT_TEXTIN, plaintext[::-1])
            
            # Run the target: start acquisition on PicoScope
            ps.runBlock()
            time.sleep(0.05)
            
            # Trigger program execution and acquisition
            cw305.fpga_write(cw305.REG_USER_LED, [0x01])
            cw305.usb_trigger_toggle()
            ps.waitReady()
            
            # Retrieve captured power trace
            trace_data = ps.getDataV()
            
            # Read ciphertext from target (endianness reversed for HW AES IP)
            ciphertext_raw = cw305.fpga_read(cw305.REG_CRYPT_CIPHEROUT, 16)
            ciphertext = ciphertext_raw[::-1]

            # Sanity check with software golden model
            formatted_pt = [format(el, '02x') for el in plaintext]
            text_int = [int(subbyte, 16) for subbyte in formatted_pt]
            expected_ct = cipher.encrypt(formatted_key, text_int, tested_sbox)
            
            assert list(ciphertext) == list(expected_ct), (
                f"Incorrect encryption result!\nGot {list(ciphertext)}\nExp {list(expected_ct)}\n"
            )
            
            if save_traces and project is not None:
                # Store trace and metadata in ChipWhisperer project
                trace_obj = Trace(np.array(trace_data), plaintext, ciphertext, key)
                project.traces.append(trace_obj)
                
            # Generate next plaintext
            _, plaintext = ktp.next()

        if save_traces and project is not None:
            project.save()
            project.close()
            print(f"[ONLINE] Capture completed. Traces saved to: {project_file}")
        else:
            print("[ONLINE] Capture completed. Traces were not saved.")

    except (Exception, KeyboardInterrupt) as e:
        print(f"\n[ERROR] Capture aborted manually or due to error: {e}")
        if 'project' in locals() and project is not None:
            try:
                project.close()
            except Exception:
                pass
        raise
        
    finally:
        print("[ONLINE] Shutting down instruments...")
        try:
            cw305.dis()
        except Exception:
            pass
        try:
            ps.dis()
        except Exception:
            pass

### Plotting power traces captured (40 samples overlapped)
Useful for debug purpose and to see if the power traces are aligned.

In [ ]:
import chipwhisperer as cw
import numpy as np
import matplotlib.pyplot as plt

try:
    proj_hw = cw.open_project(str(project_file))
    
    traces = np.array([wave for wave in proj_hw.waves])
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.set_xlabel("Sample index")
    ax.set_ylabel("Voltage (mV)")
    ax.set_title("40 power traces overlapped")

    # Loop through the first 40 traces and plot them from start to finish
    for idx, trc in enumerate(traces[:40]):
        ax.plot(
            trc * 1000,
            alpha=0.5, # Removed 'color' to let Matplotlib auto-assign colors
        )
        
    plt.show()
        
except Exception as e:
    print(f"ERROR: Failed to process the ChipWhisperer project. Details: {e}")